# Consumo da API do GitHub com Requests

Primeira requisição

In [ ]:
import requests

In [ ]:
r = requests.get('https://api.github.com/events')

In [ ]:
r

# Visualização dos dados retornados pela API do GitHub

Visualização da resposta HTTP em formato de texto

In [ ]:
r.text

Conversão da resposta JSON para objetos Python

In [ ]:
r.json()

# Definição e configuração do endpoint

Verificação do status da requisição HTTP

In [ ]:
r = requests.get('https://api.github.com/versions')
r.status_code

Visualização das versões disponíveis na API do GitHub

In [ ]:
r.json()

# Especificando a versão da API

Definição do cabeçalho para especificar a versão da API

In [ ]:
headers = {'X-GitHub-Api-Version': '2022-11-28'}

Requisição à API do GitHub com cabeçalho de versão

In [ ]:
r = requests.get('https://api.github.com/events', headers=headers)

# Extraindo dados

### Obtendo dados dos repositórios

Definição da URL para consulta aos repositórios da empresa

In [ ]:
api_base_url = 'https://api.github.com'
owner = 'amzn' # username de quem vamos extrair os dados
url = f'{api_base_url}/users/{owner}/repos'

Visualização da URL construída para a requisição

In [ ]:
url

Verificação do status da requisição aos repositórios

In [ ]:
response = requests.get(url, headers=headers)
response.status_code

Visualização dos repositórios retornados pela API do GitHub

In [ ]:
response.json()

Contagem de repositórios retornados pela API

In [ ]:
len(response.json())

### Autenticação

baseado no problema q tivemos no projeto
Utilizamos a autenticação para ampliar o limite de requisições à API do GitHub, pois, durante a extração dos repositórios da Amazon, identificamos que a requisição inicial retornava apenas 30 dos 132 repositórios disponíveis. A autenticação é uma das etapas necessárias para viabilizar a coleta completa dos dados, juntamente com a paginação.

Configuração do token de autenticação e da versão da API do GitHub

In [ ]:
access_token = 'COLOQUE_SEU_TOKEN'

headers = {
    'Authorization': f'Bearer {access_token}',
    'X-GitHub-Api-Version': '2022-11-28'
}

### Paginando os repositorios

In [ ]:
api_base_url = 'https://api.github.com'
owner = 'amzn' # username de quem vamos extrair os dados
url = f'{api_base_url}/users/{owner}/repos'

url

Implementação da paginação para coleta dos repositórios

In [ ]:
repos_list = []
for page_num in range(1, 6):
    try:
        url_page = f'{url}?page={page_num}'
        response = requests.get(url_page, headers=headers)
        repos_list.append(response.json())
    except:
        repos_list.append(None)

Visualização da lista de repositórios coletados

In [ ]:
repos_list

Verificação da quantidade de páginas consultadas

In [ ]:
len(repos_list)

Contagem de repositórios retornados na primeira página

In [ ]:
len(repos_list[0])

# Transformando os dados

### Nome dos Repositórios

Visualização da lista de repositórios coletados pela API do GitHub

In [ ]:
repos_list

Extração do nome do terceiro repositório da primeira página

In [ ]:
repos_list[0][2] ['name']

Extração e armazenamento dos nomes dos repositórios

In [ ]:
repos_name = []
for page in repos_list:
    for repo in page:
        repos_name.append(repo['name'])

Visualização dos 10 primeiros nomes de repositórios

In [ ]:
repos_name[:10]

Contagem total de nomes de repositórios coletados

In [ ]:
len(repos_name)

### Línguagens dos repositórios

Visualização da linguagem principal do segundo repositório da segunda página

In [ ]:
repos_list[1][1]['language']

Extração e armazenamento das linguagens dos repositórios

In [ ]:
repos_language = []
for page in repos_list:
    for repo in page:
        repos_language.append(repo['language'])

Contagem total de linguagens extraídas dos repositórios

In [ ]:
len(repos_language)

Visualização da lista de linguagens dos repositórios

In [ ]:
repos_language

### Criando um DataFrame

Importação da biblioteca Pandas

In [ ]:
import pandas as pd

Criação do DataFrame com nomes e linguagens dos repositórios da Amazon

In [ ]:
dados_amz = pd.DataFrame()
dados_amz ['repository_name'] = repos_name
dados_amz ['language'] = repos_language     

Visualização do DataFrame da Amazon

In [ ]:
dados_amz

#### Salvando o DataFrame

Exportação do DataFrame da Amazon para CSV na pasta data_processed

In [ ]:
dados_amz.to_csv('../data_processed/amazon.csv', index=False)

# Armazenando os Dados

### Criando Repositório com post

Definição do endpoint da API do GitHub para criação de repositório do usuário via POST

In [ ]:
api_base_url = 'https://api.github.com'
url = f'{api_base_url}/user/repos'

url

Definição do corpo JSON para criação do repositório no GitHub via POST

In [ ]:
data = {
    'name': 'linguagens-utilizadas',
    'description': 'Repositorio com as linguagens de prog da Amazon',
    'private': False
}

response = requests.post(url, json=data, headers=headers)
response.status_code

### Formato do arquivo

Importação da biblioteca Base64 para codificação e decodificação de dados.

In [ ]:
import base64 

Leitura do arquivo CSV em modo binário e codificação do conteúdo em Base64.

In [ ]:
with open('../data_processed/amazon.csv', 'rb') as file:
    file_content = file.read()

encoded_content = base64.b64encode(file_content)

### Upload de arquivo com PUT

Definição do endpoint para envio do arquivo CSV ao repositório GitHub via PUT.

In [ ]:
api_base_url = 'https://api.github.com'
username = 'hiagofaria'
repo = 'linguagens-utilizadas'
path = 'amazon.csv'

url = f'{api_base_url}/repos/{username}/{repo}/contents/{path}'
url

Envio do arquivo CSV ao repositório GitHub por meio de uma requisição PUT, utilizando conteúdo codificado em Base64.

In [ ]:
data = {
    'message': 'Adicionando um novo arquivo',
    'content': encoded_content.decode('utf-8')
}

response = requests.put(url, json=data, headers=headers)
response.status_code